# Линейная регрессия — мини-проект

Сквозной мини-проект: один датасет, решения передаются между этапами. См. docs/project_notebook.md.


In [ ]:
# Setup
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)



## Постановка задачи `(intro)`


Предскажем **медианную стоимость жилья** в Калифорнии (OpenML `california_housing`) по демографическим и географическим признакам.

**Сюжет:** загрузка → EDA → обнаружение выбросов → очистка → один train/test split → сравнение OLS / Ridge / Lasso → выбор модели → метрики, остатки и интерпретация весов на **одном** hold-out test.


## Загрузка данных `(eda)`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml

%matplotlib inline
np.random.seed(42)
sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml(name="california_housing", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
target_col = "median_house_value"
feature_cols = [c for c in df.columns if c not in (target_col, "ocean_proximity")]
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

print(f"Объектов: {len(df)}, признаков: {len(feature_cols)}")
display(df.head())
display(df.describe().round(2))



## EDA: распределения и корреляции `(viz)`


Сильнейший сигнал — `median_income`. На гистограмме цены видны **хвосты** (кандидаты в выбросы). Корреляция `total_rooms` ↔ `total_bedrooms` намекает на мультиколлинеарность.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df[target_col], bins=40, color="steelblue", edgecolor="white")
axes[0].set_xlabel("median_house_value")
axes[0].set_title("Распределение цены")

axes[1].scatter(df["median_income"], df[target_col], alpha=0.25, s=12, color="steelblue")
axes[1].set_xlabel("median_income")
axes[1].set_ylabel("median_house_value")
axes[1].set_title("Цена vs доход")

corr = df[feature_cols + [target_col]].corr()
sns.heatmap(corr, ax=axes[2], cmap="RdBu_r", center=0, vmin=-1, vmax=1)
axes[2].set_title("Корреляции")
plt.tight_layout()
plt.show()

print("Корр. total_rooms–total_bedrooms:", corr.loc["total_rooms", "total_bedrooms"].round(3))
print("Корр. median_income–цена:", corr.loc["median_income", target_col].round(3))



## Выбросы в целевой переменной `(experiment)`


MSE сильно штрафует экстремальные значения **Y**. Посмотрим boxplot и правило **IQR** для `median_house_value`.


In [ ]:
q1, q3 = df[target_col].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_mask = (df[target_col] < lower) | (df[target_col] > upper)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(y=df[target_col], ax=axes[0], color="steelblue")
axes[0].set_title("Boxplot цены")
axes[0].set_ylabel("median_house_value")

axes[1].scatter(df.loc[~outlier_mask, "median_income"], df.loc[~outlier_mask, target_col],
                alpha=0.2, s=10, color="steelblue", label="норма")
axes[1].scatter(df.loc[outlier_mask, "median_income"], df.loc[outlier_mask, target_col],
                alpha=0.8, s=25, color="crimson", label="выброс IQR")
axes[1].set_xlabel("median_income")
axes[1].set_ylabel("median_house_value")
axes[1].set_title("Выбросы по IQR")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"IQR-границы: [{lower:,.0f}; {upper:,.0f}]")
print(f"Выбросов: {outlier_mask.sum()} ({100 * outlier_mask.mean():.1f}%)")



## Решение: очистка данных `(example)`


**Решение:** удаляем объекты с выбросами по IQR в `median_house_value`. Дальнейшие шаги — только на `df_clean`.


In [ ]:
df_clean = df.loc[~outlier_mask].copy().reset_index(drop=True)
print(f"Было {len(df)} → осталось {len(df_clean)} объектов")
print(f"Медиана цены: {df[target_col].median():,.0f} → {df_clean[target_col].median():,.0f}")



## Решение: train / test split `(example)`


**Решение:** один раз делим **очищенные** данные на train и test (`random_state=42`). Этот test используем до конца ноутбука.


In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean[feature_cols].values
y = df_clean[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}, test: {len(X_test)}")



## Baseline: OLS в Pipeline `(example)`


`StandardScaler` обучаем **только на train** — внутри `Pipeline`, без утечки из test.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pipe_ols = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
pipe_ols.fit(X_train, y_train)
y_pred_ols = pipe_ols.predict(X_test)

print(f"OLS — Test MAE:  {mean_absolute_error(y_test, y_pred_ols):,.0f}")
print(f"OLS — Test RMSE: {mean_squared_error(y_test, y_pred_ols) ** 0.5:.3f}")
print(f"OLS — Test R2:   {r2_score(y_test, y_pred_ols):.3f}")



## Сравнение Ridge и Lasso `(experiment)`


Ridge (L2) сжимает веса при мультиколлинеарности. Lasso (L1) может обнулять слабые признаки. Сравниваем на **том же** train/test.


In [ ]:
from sklearn.linear_model import Ridge, Lasso

candidates = {
    "OLS": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.01, max_iter=10000))]),
}

rows = []
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    pred_te = pipe.predict(X_test)
    w = pipe.named_steps["model"].coef_
    rows.append({
        "model": name,
        "R2 test": r2_score(y_test, pred_te),
        "RMSE test": mean_squared_error(y_test, pred_te) ** 0.5,
        "norm_w": np.linalg.norm(w),
    })

metrics_df = pd.DataFrame(rows).sort_values("RMSE test")
display(metrics_df.round(4))



## Решение: выбор модели `(example)`


**Решение:** берём модель с лучшим test RMSE. Дальше все метрики и графики — только для неё.


In [ ]:
best_name = metrics_df.iloc[0]["model"]
final_pipe = candidates[best_name]
final_pipe.fit(X_train, y_train)
y_pred = final_pipe.predict(X_test)

print(f"Выбрана модель: {best_name}")
print(f"Test RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.3f}")
print(f"Test R2:   {r2_score(y_test, y_pred):.3f}")



## Диагностика остатков `(viz)`


Идеал — случайное облако вокруг нуля. Паттерн «воронки» намекает на нелинейность.


In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_pred, residuals, alpha=0.35, s=15, color="steelblue")
axes[0].axhline(0, color="crimson", linestyle="--")
axes[0].set_xlabel("предсказание ŷ")
axes[0].set_ylabel("остаток y − ŷ")
axes[0].set_title(f"Residual plot ({best_name})")

axes[1].hist(residuals, bins=35, color="steelblue", edgecolor="white")
axes[1].set_xlabel("остаток")
axes[1].set_title("Гистограмма остатков")
plt.tight_layout()
plt.show()



## Интерпретация весов `(viz)`


После масштабирования сравниваем силу признаков (**ceteris paribus** — при прочих равных).


In [ ]:
w = final_pipe.named_steps["model"].coef_
order = np.argsort(np.abs(w))

plt.figure(figsize=(8, 5))
plt.barh(np.array(feature_cols)[order], w[order], color="steelblue")
plt.xlabel("вес (StandardScaler → модель)")
plt.title(f"Интерпретация весов ({best_name}, данные без выбросов)")
plt.tight_layout()
plt.show()

for name, val in sorted(zip(feature_cols, w), key=lambda t: abs(t[1]), reverse=True)[:4]:
    print(f"{name:12s}: {val:+.3f}")



## Чек-лист мини-проекта `(summary)`


1. EDA → обнаружили выбросы → **удалили** → `df_clean`.
2. Один `train_test_split` до любых трансформаций.
3. `Pipeline` + `StandardScaler` — без утечки из test.
4. Сравнили OLS / Ridge / Lasso → **выбрали** `final_pipe`.
5. Residual plot и веса — на **финальной** модели и **том же** test.
6. Если R2 train >> R2 test — переобучение; паттерн в остатках — нелинейность.
